# Objective:   
Automatically tag support tickets into categories using a large language model (LLM). 

In [ ]:
import pandas as pd 
import numpy as np  

df=pd.read_csv("twcs.csv",nrows=10000)
#Now check shape of dataset
print(f'Shape of Dataset {df.shape}')
#View Columns in list 
print(f'columns are {df.columns.tolist()}')
#View first five rows
print(df.head())

# Due to Auto tagging support now Define Classification Categories

In [ ]:
#Now we are using dictionary containing key value pairs which LLM will used for auto tagging
CATEGORIES = {
    "billing": "Issues related to payments, refunds, charges, subscriptions, pricing",
    "technical": "Technical problems, bugs, app/website not working, errors, crashes",
    "account_access": "Login issues, password reset, account locked, profile management",
    "product_inquiry": "Questions about product features, specifications, availability",
    "delivery_shipping": "Issues with order delivery, shipping delays, lost packages",
    "return_refund": "Return requests, refund processing, exchange requests",
    "customer_service": "General complaints, poor service, agent behavior",
    "feature_request": "Suggestions for new features or improvements",
    "cancellation": "Account or subscription cancellation requests"
}

category_names=list(CATEGORIES.keys())
print(f'Category Names:{category_names}')
category_description=list(CATEGORIES.values())
print(f'Category Description: {category_description}')


# Now Setup OpenAI API 

In [ ]:
import os
from openai import OpenAI

# Now the client will find it
client = OpenAI() 
from dotenv import load_dotenv, find_dotenv
# Automatically finds the .env file in the current or parent directories
load_dotenv(find_dotenv())
# Get the key from the environment
api_key = os.getenv('OPENAI_API_KEY')
# Initialize the client
client = OpenAI(api_key=api_key)
try:
    res = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{'role': 'user', 'content': 'Hello'}],
        max_tokens=5
    )
    print("✅ Connection Successful!")
except Exception as e:
    print(f"❌ Error: {e}")

# Zero-Shot Classification Implementation

In [ ]:
def zero_shot_classify(text, categories, category_descriptions):
    """
    Classify a support ticket using zero-shot prompting.
    """
    # Build the prompt
    categories_text = "\n".join([f"- {cat}: {desc}" for cat, desc in zip(categories, category_descriptions)])
    
    prompt = f"""You are a customer support ticket classifier. Analyze the following customer message and classify it into the most appropriate category.

Categories:
{categories_text}

Customer message: "{text}"

Respond with a JSON object containing:
1. "primary_category": The main category for this ticket
2. "confidence": Your confidence score (0-1)
3. "reasoning": Brief explanation of your classification"""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a precise ticket classifier. Respond only with valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,  # Low temperature for consistent results
            response_format={"type": "json_object"}
        )
        
        import json
        result = json.loads(response.choices[0].message.content)
        return result
        
    except Exception as e:
        print(f"Error: {e}")
        return {"primary_category": "unknown", "confidence": 0, "reasoning": "Error occurred"}

# Test the zero-shot classifier
sample_text = "I was charged twice for my subscription this month. Please refund the extra charge."
result = zero_shot_classify(sample_text, category_names, category_description)
print(f"Zero-shot classification result: {result}")

# Get Top 3 most Probable Categories with Scores

In [ ]:
def zero_shot_get_top3(text, categories, category_descriptions):
    """
    Get top 3 most probable categories with confidence scores.
    """
    categories_text = "\n".join([f"- {cat}: {desc}" for cat, desc in zip(categories, category_descriptions)])
    
    prompt = f"""Analyze this customer support message and rank the top 3 most likely categories.

Categories with descriptions:
{categories_text}

Customer message: "{text}"

Respond with a JSON object containing a "predictions" array of 3 objects, each with:
- "category": The category name
- "confidence": Confidence score (0-100)
- "reason": Why this category fits

Order from most likely to least likely."""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a precise ticket classifier. Respond only with valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        
        import json
        result = json.loads(response.choices[0].message.content)
        return result.get("predictions", [])
        
    except Exception as e:
        print(f"Error: {e}")
        return []

# Test getting top 3 tags
sample_text = "My app keeps crashing when I try to upload photos. Please help!"
top3 = zero_shot_get_top3(sample_text, category_names, category_description)
print("Top 3 categories:")
for pred in top3:
    print(f"  - {pred['category']}: {pred['confidence']}% - {pred['reason']}")

# Few Shot Implementation.

In [ ]:
def create_few_shot_examples():
    """
    Create labeled examples for few-shot learning.
    """
    examples = [
        {
            "text": "I need a refund for order #12345. The product arrived damaged.",
            "category": "return_refund"
        },
        {
            "text": "I can't log into my account. It says password incorrect even after reset.",
            "category": "account_access"
        },
        {
            "text": "When will you release the new update with dark mode?",
            "category": "feature_request"
        },
        {
            "text": "My payment went through but I didn't get confirmation email.",
            "category": "billing"
        },
        {
            "text": "The app freezes every time I open the settings page.",
            "category": "technical"
        },
        {
            "text": "Cancel my subscription immediately. I don't want to be charged next month.",
            "category": "cancellation"
        }
    ]
    return examples

def few_shot_classify(text, categories, category_descriptions, examples, get_top3=False):
    """
    Classify using few-shot learning with provided examples.
    """
    # Format examples for the prompt
    examples_text = ""
    for ex in examples:
        examples_text += f"Message: \"{ex['text']}\"\nCategory: {ex['category']}\n\n"
    
    categories_text = "\n".join([f"- {cat}: {desc}" for cat, desc in zip(categories, category_descriptions)])
    
    output_instruction = "Respond with ONLY the category name" if not get_top3 else """
Respond with a JSON object containing a "predictions" array of 3 objects, each with:
- "category": The category name
- "confidence": Confidence score (0-100)
- "reason": Why this category fits
Order from most likely to least likely."""

    prompt = f"""You are a customer support ticket classifier. Here are some examples of correctly classified tickets:

{examples_text}

Now classify this new ticket using the same logic.

Categories:
{categories_text}

New message: "{text}"

{output_instruction}"""

    try:
        if get_top3:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": "You are a precise ticket classifier. Respond only with valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            import json
            result = json.loads(response.choices[0].message.content)
            return result.get("predictions", [])
        else:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": "You are a precise ticket classifier."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1
            )
            return response.choices[0].message.content.strip()
            
    except Exception as e:
        print(f"Error: {e}")
        return []

# Test few-shot classification
examples = create_few_shot_examples()

# Single category prediction
sample_text = "My order was supposed to arrive yesterday but it's still not here."
result = few_shot_classify(sample_text, category_names, category_description, examples, get_top3=False)
print(f"Few-shot classification: {result}")

# Top 3 predictions
top3_fewshot = few_shot_classify(sample_text, category_names, category_description, examples, get_top3=True)
print("\nTop 3 categories (few-shot):")
for pred in top3_fewshot:
    print(f"  - {pred['category']}: {pred['confidence']}%")

# Evaluation of Results

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time

def evaluate_classifiers(test_samples, ground_truth, categories):
    """
    Compare zero-shot and few-shot classifier performance.
    """
    results = {
        "zero_shot": {"predictions": [], "times": []},
        "few_shot": {"predictions": [], "times": []}
    }
    
    examples = create_few_shot_examples()
    
    for i, (text, true_label) in enumerate(zip(test_samples, ground_truth)):
        print(f"Processing sample {i+1}/{len(test_samples)}...")
        
        # Zero-shot
        start = time.time()
        zs_result = zero_shot_classify(text, categories, category_description)
        zs_time = time.time() - start
        results["zero_shot"]["predictions"].append(zs_result.get("primary_category", "unknown"))
        results["zero_shot"]["times"].append(zs_time)
        
        # Few-shot
        start = time.time()
        fs_result = few_shot_classify(text, categories, category_description, examples, get_top3=False)
        fs_time = time.time() - start
        results["few_shot"]["predictions"].append(fs_result)
        results["few_shot"]["times"].append(fs_time)
    
    # Calculate metrics
    for method in ["zero_shot", "few_shot"]:
        preds = results[method]["predictions"]
        valid_indices = [i for i, p in enumerate(preds) if p != "unknown" and p in categories]
        
        if valid_indices:
            valid_preds = [preds[i] for i in valid_indices]
            valid_true = [ground_truth[i] for i in valid_indices]
            
            accuracy = accuracy_score(valid_true, valid_preds)
            precision, recall, f1, _ = precision_recall_fscore_support(
                valid_true, valid_preds, average='weighted', zero_division=0
            )
            
            print(f"\n{method.upper()} Performance:")
            print(f"  Accuracy: {accuracy:.3f}")
            print(f"  Precision: {precision:.3f}")
            print(f"  Recall: {recall:.3f}")
            print(f"  F1-Score: {f1:.3f}")
            print(f"  Avg Time: {np.mean(results[method]['times']):.2f}s")

# Sample test data (create a small labeled test set manually)
test_samples = [
    "I was double charged for my membership",
    "The app crashes when I open the camera",
    "Can't reset my password, the email never arrives",
    "When will you add fingerprint login?",
    "My package shows delivered but I never received it"
]

ground_truth = ["billing", "technical", "account_access", "feature_request", "delivery_shipping"]

# Run evaluation
evaluate_classifiers(test_samples, ground_truth, category_names)

In [ ]:
def multi_class_ranking(text, categories, category_descriptions):
    """
    Predict probability for ALL categories and rank them.
    This is the key function for multi-class prediction and ranking.
    """
    categories_text = "\n".join([f"{i+1}. {cat}: {desc}" for i, (cat, desc) in enumerate(zip(categories, category_descriptions))])
    
    prompt = f"""Analyze this customer support message and provide a probability score for EVERY category.

Categories:
{categories_text}

Customer message: "{text}"

IMPORTANT: You MUST provide a score for ALL {len(categories)} categories. 
The scores should sum to 100% (or 1.0 if using decimals).

Respond with a JSON object containing:
{{
    "ranked_predictions": [
        {{"category": "category_name", "probability": score, "reasoning": "why"}},
        ... (ALL {len(categories)} categories, sorted by probability descending)
    ]
}}"""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": f"""You are a multi-class classifier. You MUST output probabilities for ALL {len(categories)} categories.
                The probabilities must sum to 100. Sort from highest to lowest probability."""},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,  # Slightly higher for better probability distribution
            response_format={"type": "json_object"}
        )
        
        import json
        result = json.loads(response.choices[0].message.content)
        
        # Validate that we have all categories
        predicted_cats = [p['category'] for p in result['ranked_predictions']]
        missing_cats = set(categories) - set(predicted_cats)
        
        if missing_cats:
            print(f"Warning: Missing categories {missing_cats}. Adding with 0 probability.")
            for cat in missing_cats:
                result['ranked_predictions'].append({
                    "category": cat,
                    "probability": 0,
                    "reasoning": "Not applicable"
                })
        
        # Sort by probability descending
        result['ranked_predictions'].sort(key=lambda x: x['probability'], reverse=True)
        
        return result['ranked_predictions']
        
    except Exception as e:
        print(f"Error: {e}")
        # Return equal probabilities as fallback
        equal_prob = 100 / len(categories)
        return [{"category": cat, "probability": equal_prob, "reasoning": "Fallback"} for cat in categories]

# Test the multi-class ranking
sample_text = "I was charged twice for my subscription. Please refund the extra payment."
all_ranked_predictions = multi_class_ranking(sample_text, category_names, category_description)

print("=" * 60)
print("MULTI-CLASS PREDICTION AND RANKING RESULTS")
print("=" * 60)
print(f"\nTicket: \"{sample_text}\"\n")
print(f"{'Rank':<6} {'Category':<20} {'Probability':<12} {'Reasoning'}")
print("-" * 60)

for rank, pred in enumerate(all_ranked_predictions, 1):
    print(f"{rank:<6} {pred['category']:<20} {pred['probability']:.1f}%      {pred['reasoning'][:40]}...")

In [ ]:
def batch_multi_class_ranking(tickets, categories, category_descriptions, batch_size=5):
    """
    Process multiple tickets with multi-class ranking.
    """
    all_results = []
    
    for i in range(0, len(tickets), batch_size):
        batch = tickets[i:i+batch_size]
        batch_results = []
        
        for ticket in batch:
            ranked = multi_class_ranking(ticket, categories, category_description)
            batch_results.append({
                "text": ticket,
                "ranked_predictions": ranked,
                "top_3": ranked[:3],
                "top_1_confidence": ranked[0]['probability'] if ranked else 0,
                "entropy": calculate_entropy([p['probability']/100 for p in ranked])
            })
        
        all_results.extend(batch_results)
        print(f"Processed {min(i+batch_size, len(tickets))}/{len(tickets)} tickets")
    
    return all_results

def calculate_entropy(probabilities):
    """
    Calculate entropy to measure prediction uncertainty.
    Higher entropy = model is less confident (more evenly distributed probabilities)
    """
    import math
    probs = [p for p in probabilities if p > 0]
    entropy = -sum(p * math.log2(p) for p in probs)
    return entropy

def analyze_multi_class_predictions(results):
    """
    Analyze the multi-class prediction quality.
    """
    print("\n" + "="*70)
    print("MULTI-CLASS PREDICTION ANALYSIS")
    print("="*70)
    
    # Calculate metrics
    top1_confidences = [r['top_1_confidence'] for r in results]
    entropies = [r['entropy'] for r in results]
    
    print(f"\n📊 Prediction Confidence Statistics:")
    print(f"   - Average Top-1 Confidence: {np.mean(top1_confidences):.1f}%")
    print(f"   - Min Top-1 Confidence: {np.min(top1_confidences):.1f}%")
    print(f"   - Max Top-1 Confidence: {np.max(top1_confidences):.1f}%")
    print(f"   - Std Dev: {np.std(top1_confidences):.1f}%")
    
    print(f"\n🎯 Prediction Certainty (Entropy - lower is more certain):")
    print(f"   - Avg Entropy: {np.mean(entropies):.2f} bits")
    print(f"   - Min Entropy: {np.min(entropies):.2f} bits (most certain)")
    print(f"   - Max Entropy: {np.max(entropies):.2f} bits (most uncertain)")
    
    # Show examples
    print(f"\n📝 Example Rankings (First 3 tickets):")
    for i, result in enumerate(results[:3]):
        print(f"\nTicket {i+1}: \"{result['text'][:80]}...\"")
        print(f"  Top-3 Categories:")
        for rank, pred in enumerate(result['top_3'], 1):
            print(f"    {rank}. {pred['category']}: {pred['probability']:.1f}%")
        print(f"  Prediction Entropy: {result['entropy']:.2f} bits")
    
    return {
        "avg_top1_confidence": np.mean(top1_confidences),
        "avg_entropy": np.mean(entropies),
        "uncertain_predictions": [r for r in results if r['entropy'] > 2.5]  # High uncertainty threshold
    }

# Test with multiple tickets
test_tickets = [
    "I was charged twice for my subscription. Please refund the extra payment.",
    "The app keeps crashing when I try to upload photos to my profile.",
    "Can you add dark mode to the mobile app? That would be great!",
    "My account was locked after too many failed login attempts.",
    "When will my order #ORD-12345 be delivered? It's been 2 weeks."
]

# Run multi-class ranking on all tickets
multi_class_results = batch_multi_class_ranking(test_tickets, category_names, category_description)

# Analyze the results
analysis = analyze_multi_class_predictions(multi_class_results)